# training-my-moe -- Interactive Code Tour

A hands-on, runnable companion to the `qwenmoe/` codebase: one notebook per
subpackage, each demoing every public class and function with real
executable code against this repo's own real artifacts (a real trained
checkpoint, real cached training data) -- no fabricated numbers, matching
this project's own documentation convention (see `journey_of_a_model/`).

For full mathematical derivations, papers, and worked-by-hand numeric
examples, see `journey_of_a_model/*.md` -- this tour intentionally stays
lighter: code + concise notes + live demos, one class/function at a time.

## Notebooks in this tour
| # | Notebook | Covers |
|---|----------|--------|
| 00 | this notebook | project map + a 60-second end-to-end tour |
| 01 | [01_config.ipynb](01_config.ipynb) | `qwenmoe/config.py`, `configs/presets.py` |
| 02 | [02_tokenizer.ipynb](02_tokenizer.ipynb) | `qwenmoe/tokenizer/` |
| 03 | [03_data.ipynb](03_data.ipynb) | `qwenmoe/data/` |
| 04 | [04_model.ipynb](04_model.ipynb) | `qwenmoe/model/` (norm, positional, attention, MoE, block, transformer, generate, trace) |
| 05 | [05_train.ipynb](05_train.ipynb) | `qwenmoe/train/` (trainer, scaling, compare) |
| 06 | [06_viz.ipynb](06_viz.ipynb) | `qwenmoe/viz/` (trace capture; Manim scenes removed) |

## How to use these notebooks
- Run cells top to bottom; select this project's `.venv` as the kernel.
- Most demos reuse real artifacts already on disk -- a real trained
  checkpoint under `runs/smoke_test/`, and real cached tiny-Shakespeare
  data under `.cache/data/` -- so almost everything below runs in well
  under a second. A couple of cells in notebooks 05/06 do a small amount
  of real training/rendering and take a few seconds.

In [13]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

import torch
import qwenmoe

print("torch version:", torch.__version__)
print("qwenmoe package location:", Path(qwenmoe.__file__).parent)

Working directory: /Users/sjoseph/Documents/Study/training-my-moe
torch version: 2.13.0
qwenmoe package location: /Users/sjoseph/Documents/Study/training-my-moe/qwenmoe


## Project map

```mermaid
flowchart LR
    CFG["config.py
    ModelConfig / TrainConfig / DataConfig"] --> TOK
    CFG --> DATA
    CFG --> MODEL
    TOK["tokenizer/
    CharTokenizer, ByteBPETokenizer"] --> DATA
    DATA["data/
    download, TokenDataset, fixed slices"] --> TRAIN
    MODEL["model/
    RMSNorm, positional, attention, MoE,
    TransformerBlock, QwenMoEModel, generate, trace"] --> TRAIN
    TRAIN["train/
    Trainer, scaling utilities, run_comparison"] --> VIZ
    MODEL --> VIZ["viz/
    capture_trace (Manim scenes removed)"]
```

In [14]:
for path in sorted(Path("qwenmoe").rglob("*.py")):
    if path.name == "__init__.py":
        continue
    print(path)

qwenmoe/config.py
qwenmoe/data/dataset.py
qwenmoe/data/download.py
qwenmoe/model/attention.py
qwenmoe/model/block.py
qwenmoe/model/generate.py
qwenmoe/model/moe.py
qwenmoe/model/positional.py
qwenmoe/model/rmsnorm.py
qwenmoe/model/trace.py
qwenmoe/model/transformer.py
qwenmoe/tokenizer/base.py
qwenmoe/tokenizer/bpe.py
qwenmoe/tokenizer/char.py
qwenmoe/train/compare.py
qwenmoe/train/scaling.py
qwenmoe/train/trainer.py
qwenmoe/viz/common.py
qwenmoe/viz/scenes_attention.py
qwenmoe/viz/scenes_embedding.py
qwenmoe/viz/scenes_moe.py
qwenmoe/viz/scenes_norm_act.py
qwenmoe/viz/scenes_output.py
qwenmoe/viz/scenes_task10_preview.py
qwenmoe/viz/scenes_task11_preview.py
qwenmoe/viz/scenes_task15_preview.py
qwenmoe/viz/scenes_task3_preview.py
qwenmoe/viz/scenes_task5_preview.py
qwenmoe/viz/scenes_task6_preview.py
qwenmoe/viz/scenes_task7_preview.py
qwenmoe/viz/scenes_task8_preview.py
qwenmoe/viz/scenes_task9_preview.py
qwenmoe/viz/scenes_training.py


## A 60-second end-to-end tour
Before going module by module, let's touch every layer once: build a tiny
model from the real `SMOKE_TEST` preset, run one forward pass, then load
this repo's real trained checkpoint and generate text with it.

In [15]:
from configs.presets import SMOKE_TEST
from qwenmoe.model.transformer import QwenMoEModel
from qwenmoe.train.scaling import count_params

torch.manual_seed(0)
model = QwenMoEModel(SMOKE_TEST)
print(SMOKE_TEST)
print(count_params(model))

ModelConfig(vocab_size=65, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=128, n_shared_experts=1, n_routed_experts=4, top_k=2, max_seq_len=128, positional_scheme='rope', activation='swiglu', moe_balancing='aux_loss', attention_variant='gqa', mla_latent_dim=None, lora_rank=None, tie_word_embeddings=True, rope_theta=10000.0)
ParamBreakdown(embedding=4224, attention_total=24704, moe_total=246400, moe_active=148096, total=275328, active=177024)


In [16]:
import torch.nn.functional as F

input_ids = torch.randint(0, SMOKE_TEST.vocab_size, (2, 16))
logits, extras = model(input_ids)
loss = F.cross_entropy(logits.reshape(-1, SMOKE_TEST.vocab_size), input_ids.reshape(-1))

print("logits shape:", logits.shape)
print("random-init loss:", loss.item())
print("ln(vocab_size)  :", torch.log(torch.tensor(float(SMOKE_TEST.vocab_size))).item())
print("(these two should be close -- a real bug in this project's task 10 was an")
print(" embedding init that blew this up to ~60 instead; see journey_of_a_model/010-training-loop.md)")

logits shape: torch.Size([2, 16, 65])
random-init loss: 4.103914737701416
ln(vocab_size)  : 4.174387454986572
(these two should be close -- a real bug in this project's task 10 was an
 embedding init that blew this up to ~60 instead; see journey_of_a_model/010-training-loop.md)


In [11]:
from qwenmoe.model.generate import generate
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint

ckpt = load_checkpoint("runs/smoke_test/checkpoints/step_1000.pt")
trained_model = QwenMoEModel(ckpt["model_config"])
trained_model.load_state_dict(ckpt["model_state_dict"])
tokenizer = build_tokenizer_from_meta(ckpt["tokenizer_meta"])

torch.manual_seed(0)
print("TRAINED MODEL (step 1000):")
print(generate(trained_model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))

TRAINED MODEL (step 1000):
ROMEO:
Thremh
Whit whonlt I doce meaman youd I tostet and fue yoXOBETI
Thee hath thigh


### Raw (untrained) vs. trained inference
`model` (built a few cells up) shares the exact same `SMOKE_TEST`
architecture and the same tokenizer, but was never trained -- its weights
are still the random initialization. Generating from it with the identical
decoding settings shows what "random weights" actually looks like as text,
in direct contrast to the trained checkpoint's output above.

In [12]:
torch.manual_seed(0)
print("RAW (untrained) MODEL:")
print(generate(model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))

RAW (untrained) MODEL:
ROMEO:w3ZrjmhPkqkARxLsX?cUWRoHca
?exhjyU&aNdkIxAlsvSBUMycpRD$By$O-Pntw
kVQMOh&?j DJR!s


## Where to go next
Head to [01_config.ipynb](01_config.ipynb) to start at the bottom of the
stack, or jump straight to whichever module you're curious about using the
table above. For the full spec this project was built from, see
`.kiro/specs/qwen-moe-from-scratch/{requirements,design,tasks}.md`.